This notebook runs various smaller-scale experiments on the MNIST dataset that is padded to allow larger transformations.

This includes a comparison of whether restricting the domain has advantages by comparing different values for the boundary that are multiples of defualt domain choosen. (In this example it does)

Whether changing the size of the search domain is advantageous. (Smaller can benefit)

Which sampling method performed best (uniformly spaced sampling (lattice) for coordinate descent, Sobol for random search).

An experiment that shows that reflecting at the borders is better than simply setting it to the domain boundary during the search.

The last experiment adds translation to the transformations searched even though it was not in the original set showing a degredation in accuracy.

In [ ]:
import torch
import torchvision
from matplotlib import pyplot as plt
from src.search.coordinate_descent import CoordinateDescent, WeightedCoordinateDescent
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#print pytorch version
print(torch.__version__)

In [ ]:
#look for experiment files in parents
import os
path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)


In [ ]:
experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")

In [ ]:
dataset ="bigger_mnist"
default_architecutre_mapping = {
    "mnist":"resnet_small",
    "bigger_mnist":"resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist":"bigger_extended_resnet_small",
    "coil100":"coil_resnet_small",
    "tu_berlin":"bi_lstm",
    "modelnet10":"pointnetplus",

}



architecture = default_architecutre_mapping[dataset]

In [ ]:
from src.data.get_dataset import get_dataset_info,get_dataset

dataset_info = get_dataset_info(dataset)
dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data)


In [ ]:
dataset_info

In [ ]:
dataset_dict.keys()

In [ ]:
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']



In [ ]:
batch_size = next(iter(train_loader))[0].shape[0]

In [ ]:
#test images
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(10, 5))

axs[0].imshow(torchvision.utils.make_grid(next(iter(train_loader))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[1].imshow(torchvision.utils.make_grid(next(iter(val_loader))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[2].imshow(torchvision.utils.make_grid(next(iter(test_loader_transformed))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[0].set_title('Training')
axs[1].set_title('Validation')
axs[2].set_title('Test')

for ax in axs.flat:
    ax.axis('off')

In [ ]:
from model.train import train_and_get_model
from model.get_model import get_network


from src.utils.eval.main_model import evaluate_base_model

In [ ]:
model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")

In [ ]:
model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": 100,
        "precision": "16-mixed",
},load_if_exists=True)


res = evaluate_base_model(model, test_loader_transformed, device)
print(res)

In [ ]:
from data.transformation import get_transformation_sequence_images
# get transform sequence
if dataset_info.transform_seq_name is not None and dataset_info.datatype == "image":
    transform_seq = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method
    ).cuda()

In [ ]:
model.eval().cuda()

In [ ]:
# get emeddinb cache. The purpose is to caceh embeddings so they dont have to be recomputed.
from torch.utils.data import SequentialSampler
from embedding_cache import LayerEmbeddingCache
transform_name = dataset_info.transform_seq_name

cache_name_train = f"{dataset}_{architecture}_{transform_name}_embedding_cache_train"

cache_train = LayerEmbeddingCache(model,train_loader_no_shuffle,cache_dir=os.path.join(embedding_cache_path,cache_name_train))
embeddings_t,final_t,classes_t = cache_train.__call__("fc",capture_modes='input',flatten=True)

In [ ]:
#the cache also provides a helper to make the model return a pair of the intermediate output(embeddings) and the final output
dual_output_model = cache_train.make_wrapper("fc",capture_modes='input',concat=False,flatten=True)


In [ ]:
from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence, PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence,PerClassKNNConfidence

from confidence.input_transform import InputTransformImage,PCAInputModule,RandomProjectionModule
#using this we can create confidence module. This has to be fitted on the embeddings and also uses an optional reducer to decrease the dim of the embeddings.

input_transform_image = InputTransformImage((3,3),(128,7,7))
input_transform_pca = PCAInputModule(512)
input_transform_random = RandomProjectionModule(512,method="gaussian")


nn_pytorch_pretrained = PerClassKNNConfidence(metric="cosine",input_transform=None)
nn_pytorch_pretrained.fit(embeddings_t,classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained,EnergyConfidence(), mult=False,b=0.0)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model,conf_split_pretrained,index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq,consolidate_method="consolidate_simple")


In [ ]:
from search.gradient_descent import ParallelGradientDescent
from search.simulated_anealing import ParallelSimulatedAnnealing

In [ ]:
repeats=10
overwrite = False

In [ ]:
from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search

ps = ParallelSimulatedAnnealing(max_iterations=12,initial_temp=0.0,neighbor_hood_size=0.25,project_param=True,parallel_runs=10)
conf_energy= SinglePassConfidence(model, EnergyConfidence())
problem_energy= TransformationProblem(conf_energy, transform_seq, consolidate_method="consolidate_simple")

path = os.path.join(current_path, "experiment_files", "results",f"{dataset}","experiment_restrictive")
modelname = f"{architecture}_restricted_domain_psa_default"
complete_path = os.path.join(path, modelname + ".json")

load_or_run_evaluate_confidence_and_search(model, ps, problem_energy, test_loader_transformed, max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
ps = ParallelSimulatedAnnealing(max_iterations=12,initial_temp=0.0,neighbor_hood_size=0.25,project_param=False,parallel_runs=10)
conf_energy= SinglePassConfidence(model, EnergyConfidence())
problem_energy= TransformationProblem(conf_energy, transform_seq, consolidate_method="consolidate_simple")

complete_path = os.path.join(path, f"{architecture}_unrestricted_domain_psa_default" + ".json")
load_or_run_evaluate_confidence_and_search(model, ps, problem_energy, test_loader_transformed,
                                           max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)


In [ ]:
from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search

ps = ParallelSimulatedAnnealing(max_iterations=24,initial_temp=0.0,neighbor_hood_size=0.25,project_param=True,parallel_runs=20)
conf_energy= SinglePassConfidence(model, EnergyConfidence())
problem_energy= TransformationProblem(conf_energy, transform_seq, consolidate_method="consolidate_simple")

path = os.path.join(current_path, "experiment_files", "results",f"{dataset}","experiment_restrictive")
modelname = f"{architecture}_restricted_domain_psa_default_480"
complete_path = os.path.join(path, modelname + ".json")

load_or_run_evaluate_confidence_and_search(model, ps, problem_energy, test_loader_transformed, max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
ps = ParallelSimulatedAnnealing(max_iterations=24,initial_temp=0.0,neighbor_hood_size=0.25,project_param=False,parallel_runs=20)
conf_energy= SinglePassConfidence(model, EnergyConfidence())
problem_energy= TransformationProblem(conf_energy, transform_seq, consolidate_method="consolidate_simple")

complete_path = os.path.join(path, f"{architecture}_unrestricted_domain_psa_default_480" + ".json")
load_or_run_evaluate_confidence_and_search(model, ps, problem_energy, test_loader_transformed,
                                           max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)


In [ ]:
#now for knn 480
ps = ParallelSimulatedAnnealing(max_iterations=24,initial_temp=0.0,neighbor_hood_size=0.25,project_param=True,parallel_runs=20)
problem_energy= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq,
consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_restricted_domain_psa_knn_480" + ".json")
load_or_run_evaluate_confidence_and_search(model, ps, problem_energy, test_loader_transformed,
                                           max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)


In [ ]:
ps = ParallelSimulatedAnnealing(max_iterations=24,initial_temp=0.0,neighbor_hood_size=0.25,project_param=False,parallel_runs=20)
problem_energy= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq,
consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_unrestricted_domain_psa_knn_480" + ".json")
load_or_run_evaluate_confidence_and_search(model, ps, problem_energy, test_loader_transformed,
                                           max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
ps = ParallelSimulatedAnnealing(max_iterations=12,initial_temp=0.0,neighbor_hood_size=0.25,project_param=True,parallel_runs=10)

problem_energy= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq,
consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_restricted_domain_psa_knn" + ".json")

load_or_run_evaluate_confidence_and_search(model, ps, problem_energy, test_loader_transformed,
                                           max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)


In [ ]:
ps = ParallelSimulatedAnnealing(max_iterations=12,initial_temp=0.0,neighbor_hood_size=0.25,project_param=False,parallel_runs=10)

problem_energy= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq,
consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_unrestricted_domain_psa_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, ps, problem_energy, test_loader_transformed,
                                           max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
from search.random_search import RSLR

random_search = RSLR(initial_samples=120, local_runs=1, local_max_steps=0,project_param=True)

In [ ]:
transform_default_sequence = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method
    ).cuda()

transform_bigger = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method
    ).cuda()
#increase size of domains except first
def recursive_multiply_domains(nested_tuple, factor):
    if isinstance(nested_tuple, tuple):
        return tuple(recursive_multiply_domains(item, factor) for item in nested_tuple)
    else:
        return nested_tuple * factor

transform_bigger.domains
for i in range(1,len(transform_bigger.domains)):
    transform_bigger.domains[i] = recursive_multiply_domains(transform_bigger.domains[i], 1.25)

transform_smaller = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method
    ).cuda()
#decrease size of domains except first
for i in range(1,len(transform_smaller.domains)):
    transform_smaller.domains[i] = recursive_multiply_domains(transform_smaller.domains[i], 0.75)

true_transform = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method
    ).cuda()
true_transform.invert = True

transform_slighly_smaller = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method
    ).cuda()
#decrease size of domains except first
for i in range(1,len(transform_slighly_smaller.domains)):
    transform_slighly_smaller.domains[i] = recursive_multiply_domains(transform_slighly_smaller.domains[i], 0.9)


In [ ]:
#now eval random search with different domains


In [ ]:
complete_path = os.path.join(path, f"{architecture}_random_search_default_domain_knn2" + ".json")
problem_knn= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_default_sequence,
consolidate_method="consolidate_simple")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_knn, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
complete_path = os.path.join(path, f"{architecture}_random_search_default_domain_knn" + ".json")
problem_knn= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_default_sequence,
consolidate_method="consolidate_simple")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_knn, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:



problem_knn_bigger= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_bigger,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_bigger_domain_shgo_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_knn_bigger, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)


In [ ]:
problem_knn_smaller= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_smaller,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_smaller_domain_shgo_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_knn_smaller, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
problem_knn_slighly_smaller= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_slighly_smaller,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_slighly_smaller_domain_shgo_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_knn_slighly_smaller, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
problem_knn_true= TransformationProblem(conf_mod_nn_pytorch_pretrained,true_transform,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_true_transform_shgo_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_knn_true, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
#now repeat for energy all of them
complete_path = os.path.join(path, f"{architecture}_random_search_default_domain_energy" + ".json")
problem_energy= TransformationProblem(conf_energy,transform_default_sequence,
consolidate_method="consolidate_simple")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_energy, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
problem_energy_bigger= TransformationProblem(conf_energy,transform_bigger,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_bigger_domain_shgo_energy" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_energy_bigger, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)


In [ ]:
problem_energy_smaller= TransformationProblem(conf_energy,transform_smaller,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_smaller_domain_shgo_energy" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_energy_smaller, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
problem_energy_slighly_smaller= TransformationProblem(conf_energy,transform_slighly_smaller,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_slighly_smaller_domain_shgo_energy" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_energy_slighly_smaller, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
problem_energy_true= TransformationProblem(conf_energy,true_transform,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_true_transform_shgo_energy" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_energy_true, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
from src.utils.eval.vis import load_json, extract_per_run_metrics, summarize_runs, choose_accuracy_metric, plot_group_short, \
    build_and_write_latex, plt_setup_paper
import os
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('default')
plt_setup_paper()
base_results_dir = path


# Define the four groups explicitly (only these will be plotted)
groups = [
    # PSA (Energy)
    ("psa_energy", "PSA (Energy)", [
        ("PSA\n restricted", f"{architecture}_restricted_domain_psa_default.json"),
        ("PSA\n unrestricted", f"{architecture}_unrestricted_domain_psa_default.json"),
    ]),
    # PSA (PC-KNN)
    ("psa_knn", "PSA (PC-KNN)", [
        ("PSA\n restricted", f"{architecture}_restricted_domain_psa_knn.json"),
        ("PSA\n unrestricted", f"{architecture}_unrestricted_domain_psa_knn.json"),
    ]),

    ("psa_energy_480", "PSA 480 (Energy)", [
        ("PSA\n restricted", f"{architecture}_restricted_domain_psa_default_480.json"),
        ("PSA\n unrestricted", f"{architecture}_unrestricted_domain_psa_default_480.json"),
    ]),
    # PSA (PC-KNN) 480
    ("psa_knn_480", "PSA 480 (PC-KNN)", [
        ("PSA\n restricted", f"{architecture}_restricted_domain_psa_knn_480.json"),
        ("PSA\n unrestricted", f"{architecture}_unrestricted_domain_psa_knn_480.json"),
    ]),
    # Random search (Energy)
    ("random_energy", "Random search (Energy)", [
        ("Factor 1", f"{architecture}_random_search_default_domain_energy.json"),
        ("Factor 1.25", f"{architecture}_bigger_domain_shgo_energy.json"),
        ("Factor 0.75", f"{architecture}_smaller_domain_shgo_energy.json"),
        ("Factor 0.9", f"{architecture}_slighly_smaller_domain_shgo_energy.json"),
        ("Inverted", f"{architecture}_true_transform_shgo_energy.json"),
    ]),
    # Random search (PC-KNN)
    ("random_knn", "Random search (PC-KNN)", [
        ("Factor 1", f"{architecture}_random_search_default_domain_knn.json"),
        ("Factor 1.25", f"{architecture}_bigger_domain_shgo_knn.json"),
        ("Factor 0.75", f"{architecture}_smaller_domain_shgo_knn.json"),
        ("Factor 0.9", f"{architecture}_slighly_smaller_domain_shgo_knn.json"),
        ("Inverted", f"{architecture}_true_transform_shgo_knn.json"),
    ]),
]
W = plt_setup_paper()

#different method that allows changing the factor. To keep the figures small i export different version, in case we need to change it for the paper.

def process_and_plot_groups(groups, base_results_dir, architecture,save_paths=None,W=W,rot=True, factor=0.4):
    """
    Process and plot each group in `groups`.
    Each group is (group_key, group_title, items) where items is list of (label, filename).
    Files are looked up under `base_results_dir`.
    """
    counter =0
    for group_key, group_title, items in groups:
        # keep only files that exist, in the declared order
        existing = [(lbl, os.path.join(base_results_dir, fname)) for (lbl, fname) in items
                    if os.path.isfile(os.path.join(base_results_dir, fname))]

        # load and summarize
        summaries = {}
        for lbl, fpath in existing:
            obj = load_json(fpath)
            rows = extract_per_run_metrics(obj)
            mean, se = summarize_runs(rows)
            summaries[lbl] = {"mean": mean, "se": se}

        # choose plotting metric from intersection (prefer accuracy-like)
        common = None
        for s in summaries.values():
            cols = set(s["mean"].index.tolist())
            common = cols if common is None else (common & cols)
        if not common:
            print(f"[{group_key}] No common numeric metrics. Skipping plot.")
            continue
        metric = choose_accuracy_metric(sorted(common)) or sorted(common)[0]
        #make firts letter large

        labels = [lbl for (lbl, _) in existing]
        means = [summaries[lbl]["mean"].get(metric, np.nan) for lbl in labels]
        ses   = [summaries[lbl]["se"].get(metric, 0.0)*1.96 for lbl in labels]
        #set default figsize
        plt.rcParams["figure.figsize"] = (W,W*factor)

        # plot
        if save_paths is not None:
            save_path = save_paths[counter]
            plot_group_short(f"", labels, means, ses, ylabel=metric.capitalize(),
                             save_path=save_path,rot=rot)
        else:
            plot_group_short(f"", labels, means, ses, ylabel=metric.capitalize(),rot=rot)

        # LaTeX (full common metrics)
        build_and_write_latex(base_results_dir, architecture,
                              group_key, group_title, labels, summaries)
        counter +=1


# Example call (keeps original variables)
figure_path = os.path.join(current_path,"experiment_files","export","fig2","restrictive_vs_unrestrictive")
if not os.path.exists(figure_path):
    os.makedirs(figure_path)
save_paths_pdf = [
    os.path.join(figure_path, f"{architecture}_psa_energy.pdf"),
    os.path.join(figure_path, f"{architecture}_psa_knn.pdf"),
    os.path.join(figure_path, f"{architecture}_psa_energy_480.pdf"),
    os.path.join(figure_path, f"{architecture}_psa_knn_480.pdf"),
    os.path.join(figure_path, f"{architecture}_random_energy.pdf"),
    os.path.join(figure_path, f"{architecture}_random_knn.pdf"),
]
save_paths_pgf = [
    os.path.join(figure_path, f"{architecture}_psa_energy.pgf"),
    os.path.join(figure_path, f"{architecture}_psa_knn.pgf"),
    os.path.join(figure_path, f"{architecture}_psa_energy_480.pgf"),
    os.path.join(figure_path, f"{architecture}_psa_knn_480.pgf"),
    os.path.join(figure_path, f"{architecture}_random_energy.pgf"),
    os.path.join(figure_path, f"{architecture}_random_knn.pgf"),
]
process_and_plot_groups(groups, base_results_dir, architecture, save_paths=save_paths_pdf)
process_and_plot_groups(groups, base_results_dir, architecture, save_paths=save_paths_pgf)

save_paths_pdf_small = [
    os.path.join(figure_path, f"{architecture}_psa_energy_small.pdf"),
    os.path.join(figure_path, f"{architecture}_psa_knn_small.pdf"),
    os.path.join(figure_path, f"{architecture}_psa_energy_480_small.pdf"),
    os.path.join(figure_path, f"{architecture}_psa_knn_480_small.pdf"),
    os.path.join(figure_path, f"{architecture}_random_energy_small.pdf"),
    os.path.join(figure_path, f"{architecture}_random_knn_small.pdf"),
]






In [ ]:
process_and_plot_groups(groups, base_results_dir, architecture, save_paths=save_paths_pdf_small,W=W/2,rot=False, factor=0.6)

The following compares the different sampling algorithm once with Coordinate descent ones with random search.

In [ ]:
even_smaller = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method
    ).cuda()
#decrease size of domains except first
for i in range(1,len(even_smaller.domains)):
    even_smaller.domains[i] = recursive_multiply_domains(even_smaller.domains[i], 0.65)

In [ ]:

problem_knn_even_smaller= TransformationProblem(conf_mod_nn_pytorch_pretrained,even_smaller,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_even_smaller_domain_shgo_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_knn_even_smaller, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
#test wether to use sobol sampling or not
transform_seq_sobol = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method,
    init_method="sobol"
    ).cuda()
transform_seq_individual = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method,
    init_method="individual"
    ).cuda()

transform_seq_latin_hypercube = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method,
    init_method="latin_hypercube"
    ).cuda()

transform_seq_permuted_lattice = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method,
    init_method="permuted_lattice"
    ).cuda()

In [ ]:
#compare them
problem_knn_sobol= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq_sobol,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_sobol_sampling_shgo_knn" + ".json")

load_or_run_evaluate_confidence_and_search(model, random_search, problem_knn_sobol, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)



In [ ]:
problem_knn_individual= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq_individual,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_individual_sampling_shgo_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_knn_individual, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)


In [ ]:
problem_knn_latin_hypercube= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq_latin_hypercube,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_latin_hypercube_sampling_shgo_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_knn_latin_hypercube, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
problem_knn_permuted_lattice= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq_permuted_lattice,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_permuted_lattice_sampling_shgo_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_knn_permuted_lattice, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
groups_new = [
    # Random search (PC-KNN) with different samplings
    ("random_knn_sampling", "", [
        ("Sobol\nsampling", f"{architecture}_sobol_sampling_shgo_knn.json"),
        ("Uniform\nsampling", f"{architecture}_individual_sampling_shgo_knn.json"),
        ("Latin\nHypercube", f"{architecture}_latin_hypercube_sampling_shgo_knn.json"),
    ]),
]
save_paths_new_pdf = [
    os.path.join(figure_path, f"{architecture}_random_knn_sampling.pdf"),
]
save_paths_new_pgf = [
    os.path.join(figure_path, f"{architecture}_random_knn_sampling.pgf"),
]
process_and_plot_groups(groups_new, base_results_dir, architecture, save_paths=save_paths_new_pdf,rot =False)
process_and_plot_groups(groups_new, base_results_dir, architecture, save_paths=save_paths_new_pgf,rot =False)

In [ ]:
cd = WeightedCoordinateDescent(samples_per_dim=[12,3,3,3,3],rounds=2)

In [ ]:
#compare them
print("Starting sobol sampling for cd")
problem_knn_sobol= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq_sobol,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_cd_sobol_sampling_shgo_knn" + ".json")

load_or_run_evaluate_confidence_and_search(model, cd, problem_knn_sobol, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)



In [ ]:
print("Starting individual sampling for cd")
problem_knn_individual= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq_individual,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_cd_individual_sampling_shgo_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, cd, problem_knn_individual, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)


In [ ]:
print("Starting latin hypercube sampling for cd")
problem_knn_latin_hypercube= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq_latin_hypercube,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_cd_latin_hypercube_sampling_shgo_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, cd, problem_knn_latin_hypercube, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
print("Starting permuted lattice sampling for cd")
problem_knn_permuted_lattice= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq_permuted_lattice,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_cd_permuted_lattice_sampling_shgo_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, cd, problem_knn_permuted_lattice, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
#plot the results of cd
groups_cd = [
    # Coordinate Descent (PC-KNN) with different samplings
    ("cd_knn_sampling", "", [
        ("Sobol\nsampling", f"{architecture}_cd_sobol_sampling_shgo_knn.json"),
        ("Uniform\nsampling", f"{architecture}_cd_individual_sampling_shgo_knn.json"),
        ("Latin\nHypercube", f"{architecture}_cd_latin_hypercube_sampling_shgo_knn.json"),
        ("Lattice\nsampling", f"{architecture}_cd_permuted_lattice_sampling_shgo_knn.json"),
    ]),
]
save_paths_cd_pdf = [
    os.path.join(figure_path, f"{architecture}_cd_knn_sampling.pdf"),
]
save_paths_cd_pgf = [
    os.path.join(figure_path, f"{architecture}_cd_knn_sampling.pgf"),
]
process_and_plot_groups(groups_cd, base_results_dir, architecture, save_paths=save_paths_cd_pdf,rot=False)
process_and_plot_groups(groups_cd, base_results_dir, architecture, save_paths=save_paths_cd_pgf,rot=False)
process_and_plot_groups(groups_cd, base_results_dir, architecture,rot=False)


Now we compare the effect of reduced precision.

In [ ]:
random_search_no_amp = RSLR(initial_samples=46, local_runs=2, local_max_steps=3,project_param=False)
random_search_amp = random_search_no_amp
print()
complete_path = os.path.join(path, f"{architecture}_random_search_default_domain_amp_knn" + ".json")
with torch.amp.autocast('cuda'):
    load_or_run_evaluate_confidence_and_search(model, random_search_amp, problem_knn, test_loader_transformed
                                                 ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
print()
complete_path = os.path.join(path, f"{architecture}_random_search_default_domain_amp_energy" + ".json")
with torch.amp.autocast('cuda'):
    load_or_run_evaluate_confidence_and_search(model, random_search_amp, problem_energy, test_loader_transformed
                                                 ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
#combare results of random search default without autocast and with amp


complete_path = os.path.join(path, f"{architecture}_random_search_default_domain_no_amp_knn" + ".json")
print()
load_or_run_evaluate_confidence_and_search(model, random_search_no_amp, problem_knn, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:

complete_path = os.path.join(path, f"{architecture}_random_search_default_domain_no_amp_energy" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search_no_amp, problem_energy, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)


In [ ]:
class autocastmodelwrapper(torch.nn.Module):
    def __init__(self,model):
        super(autocastmodelwrapper, self).__init__()
        self.model = model
    def forward(self,x):
        with torch.amp.autocast('cuda'):
            return self.model(x)
dual_output_model_ac = cache_train.make_wrapper("fc",capture_modes='input',concat=False,flatten=True)
dual_output_model_ac = autocastmodelwrapper(dual_output_model_ac)
knn_conf_amp_forward_only= SinglePassConfidence(dual_output_model_ac,conf_split_pretrained,index=1)
problem_knn_amp= TransformationProblem(knn_conf_amp_forward_only,transform_seq,consolidate_method="consolidate_simple")





In [ ]:
complete_path = os.path.join(path, f"{architecture}_random_search_default_domain_amp_forward_only_knn" + ".json")

load_or_run_evaluate_confidence_and_search(model, random_search_amp, problem_knn_amp, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)



In [ ]:
energy_conf_amp_forward_only= SinglePassConfidence(autocastmodelwrapper(model),EnergyConfidence())
problem_energy_amp= TransformationProblem(energy_conf_amp_forward_only,transform_seq,consolidate_method="consolidate_simple")


In [ ]:
complete_path = os.path.join(path, f"{architecture}_random_search_default_domain_amp_forward_only_energy" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search_amp, problem_energy_amp, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)

In [ ]:
#plot the results
groups_amp = [
    # Random search (PC-KNN) with and without AMP
    ("random_knn_amp", "", [
        ("Without AMP", f"{architecture}_random_search_default_domain_no_amp_knn.json"),
        ("With AMP", f"{architecture}_random_search_default_domain_amp_knn.json"),
        ("With AMP (Model Forward only)", f"{architecture}_random_search_default_domain_amp_forward_only_knn.json"),
    ]),
    # Random search (Energy) with and without AMP
    ("random_energy_amp", "", [
        ("Without AMP", f"{architecture}_random_search_default_domain_no_amp_energy.json"),
        ("With AMP", f"{architecture}_random_search_default_domain_amp_energy.json"),
        ("With AMP (Model Forward only)", f"{architecture}_random_search_default_domain_amp_forward_only_energy.json"),
    ]),
]
save_paths_amp_pdf = [
    os.path.join(figure_path, f"{architecture}_random_knn_amp.pdf"),
    os.path.join(figure_path, f"{architecture}_random_energy_amp.pdf"),
    os.path.join(figure_path, f"{architecture}_random_energy_amp_forward_only.pdf"),
]
save_paths_amp_pgf = [
    os.path.join(figure_path, f"{architecture}_random_knn_amp.pgf"),
    os.path.join(figure_path, f"{architecture}_random_energy_amp.pgf"),
    os.path.join(figure_path, f"{architecture}_random_energy_amp_forward_only.pgf"),
]
process_and_plot_groups(groups_amp, base_results_dir, architecture, save_paths=save_paths_amp_pdf)
process_and_plot_groups(groups_amp, base_results_dir, architecture, save_paths=save_paths_amp_pgf)

In [ ]:
#print dtype of previos knn
print("Dtype of previous knn:",nn_pytorch_pretrained.dtype)

In [ ]:
#tesst wether knn with float 32 improves results
nn_pytorch_pretrained_fp32= PerClassKNNConfidence(metric="cosine",input_transform=None,dtype=torch.float32)
nn_pytorch_pretrained_fp32.fit(embeddings_t,classes_t)
nn_pytorch_pretrained_fp32.cuda()

conf_split_pretrained_fp32 = PredictedSplitConfidence(nn_pytorch_pretrained_fp32,EnergyConfidence(), mult=False,b=0.0)
conf_mod_nn_pytorch_pretrained_fp32 = SinglePassConfidence(dual_output_model,conf_split_pretrained_fp32,index=1)
problem_nn_pytorch_pretrained_fp32 = TransformationProblem(conf_mod_nn_pytorch_pretrained_fp32,transform_seq,consolidate_method="consolidate_simple")

In [ ]:
print("Dtype of knn with float32:",nn_pytorch_pretrained_fp32.dtype)

In [ ]:
#test it
complete_path = os.path.join(path, f"{architecture}_random_search_default_domain_fp32_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, random_search, problem_nn_pytorch_pretrained_fp32, test_loader_transformed
                                             ,max_batch_override=1280,repeats=repeats,save_path=complete_path,overwrite=overwrite)


In [ ]:
groups_fp32 = [
    # Random search (PC-KNN) with float32 and float16
    ("random_knn_fp32_vs_fp16", "", [
        ("PC-kNN\nFloat16", f"{architecture}_random_search_default_domain_knn.json"),
        ("PC-kNN\nFloat32", f"{architecture}_random_search_default_domain_fp32_knn.json"),
    ]),
]
save_paths_fp32_pdf = [
    os.path.join(figure_path, f"{architecture}_random_knn_fp32_vs_fp16.pdf"),
]
save_paths_fp32_pgf = [
    os.path.join(figure_path, f"{architecture}_random_knn_fp32_vs_fp16.pgf"),
]
process_and_plot_groups(groups_fp32, base_results_dir, architecture, save_paths=save_paths_fp32_pdf,rot = False)
process_and_plot_groups(groups_fp32, base_results_dir, architecture, save_paths=save_paths_fp32_pgf,rot = False)

Now we compare Gradient descent as the search when using reflection vs when clipping at the domain borders.

In [ ]:
from search.gradient_descent import ParallelGradientDescent

pgd = ParallelGradientDescent(max_iterations=5, learning_rate=0.2, project_param=True, parallel_runs=6)
pgd_no_reflect = ParallelGradientDescent(max_iterations=5, learning_rate=0.2, project_param=True, parallel_runs=6,reflect=False)

In [ ]:
#compare them
print("Starting pgd with reflection")
problem_knn= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_pgd_reflection_shgo_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, pgd, problem_knn, test_loader_transformed
                                             ,max_batch_override=1280,repeats=4,save_path=complete_path,overwrite=overwrite)

In [ ]:
print("Starting pgd without reflection")
problem_knn= TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq,
                                               consolidate_method="consolidate_simple")
complete_path = os.path.join(path, f"{architecture}_pgd_no_reflection_shgo_knn" + ".json")
load_or_run_evaluate_confidence_and_search(model, pgd_no_reflect, problem_knn, test_loader_transformed
                                             ,max_batch_override=1280,repeats=4,save_path=complete_path,overwrite=overwrite)
#

In [ ]:
#plot them
groups_pgd = [
    # Parallel Gradient Descent (PC-KNN) with and without reflection
    ("pgd_knn_reflection", "", [
        ("With reflection", f"{architecture}_pgd_reflection_shgo_knn.json"),
        ("Without reflection", f"{architecture}_pgd_no_reflection_shgo_knn.json"),
    ]),
]
save_paths_pgd_pdf = [
    os.path.join(figure_path, f"{architecture}_pgd_knn_reflection.pdf"),
]
save_paths_pgd_pgf = [
    os.path.join(figure_path, f"{architecture}_pgd_knn_reflection.pgf"),
]
process_and_plot_groups(groups_pgd, base_results_dir, architecture, save_paths=save_paths_pgd_pdf)
process_and_plot_groups(groups_pgd, base_results_dir, architecture, save_paths=save_paths_pgd_pgf)

Last experiment compares what happens when adding translation as an additional transformation not applied to original set.

In [ ]:
import os
import torch
from src.utils.transform_sequence import TransformSequence
from src.utils.affine_transforms import AffineTransformation2D

# 1) Define Transformations and Domains
transformations_trans_shearscale = [
    AffineTransformation2D.ROTATION.value,
    AffineTransformation2D.SHEARING_X.value,
    AffineTransformation2D.SHEARING_Y.value,
    AffineTransformation2D.SCALING_X.value,
    AffineTransformation2D.SCALING_Y.value,
    AffineTransformation2D.TRANSLATION.value,
]

domains_trans_shearscale = [
    (-torch.pi, torch.pi),        # angle rotation
    ((-0.5, 0.5),),               # shear x
    ((-0.5, 0.5),),               # shear y
    ((1 / 1.8 - 1, 0.8),),        # scale x delta
    ((1 / 1.8 - 1, 0.8),),        # scale y delta
    ((-0.1, 0.1), (-0.1, 0.1)),   # translation x,y
]

transform_seq_trans_shearscale = TransformSequence(
    transformations=transformations_trans_shearscale,
    domains=domains_trans_shearscale,
    device=device,
    dtype=torch.float32,
    reflect=False,
)

# 2) Setup Problems
problem_trans_shearscale = TransformationProblem(
    conf_mod_nn_pytorch_pretrained,
    transform_seq_trans_shearscale,
    consolidate_method="consolidate_simple",
    max_batch_size=1280
)

problem_knn = TransformationProblem(
    conf_mod_nn_pytorch_pretrained,
    transform_default_sequence,
    consolidate_method="consolidate_simple",
    max_batch_size=1280
)

# 3) Define distinct save paths for the JSON results
path_knn_default = os.path.join(path, f"{architecture}_random_search_default_domain_knn.json")
path_knn_trans_shearscale = os.path.join(path, f"{architecture}_random_search_trans_shearscale_knn.json")

# 4) Run evaluations
# Run default sequence
load_or_run_evaluate_confidence_and_search(
    model, random_search, problem_knn, test_loader_transformed,
    max_batch_override=1280, repeats=repeats, save_path=path_knn_default, overwrite=overwrite
)

# Run translation + shearscale sequence
load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_trans_shearscale,
    test_loader=test_loader_transformed, max_batch_override=1280,
    repeats=repeats, overwrite=overwrite, save_path=path_knn_trans_shearscale
)

In [ ]:
# Group the default setup against the trans_shearscale setup for comparison
groups_comparison = [
    ("knn_domain_comparison", "", [
        ("Default Domain", f"{architecture}_random_search_default_domain_knn.json"),
        ("Default + Transl.", f"{architecture}_random_search_trans_shearscale_knn.json"),
    ]),
]

# Define output paths for both PDF and PGF formats
save_paths_comparison_pdf = [
    os.path.join(figure_path, f"{architecture}_random_search_domain_comparison.pdf"),
]
save_paths_comparison_pgf = [
    os.path.join(figure_path, f"{architecture}_random_search_domain_comparison.pgf"),
]

# Generate the plots
process_and_plot_groups(groups_comparison, base_results_dir, architecture, save_paths=save_paths_comparison_pdf)
process_and_plot_groups(groups_comparison, base_results_dir, architecture, save_paths=save_paths_comparison_pgf)